<a href="https://colab.research.google.com/github/manu93jadon/CI-CD-Project/blob/master/IPL_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Download dataset and load in df

In [285]:
import pandas as pd
import urllib.request
url = 'https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/158/850/original/IPL_2008-2024.csv?1760202529'
output_file = 'ipldata.csv'

# Download the file
urllib.request.urlretrieve(url, output_file)
print(f"Downloaded data to {output_file}")

ipl_df = pd.read_csv(output_file)

Downloaded data to ipldata.csv


#Import Section

In [286]:
import numpy as np

# 1 ) Data Preprocessing & Feature Engineering
Prepare the IPL dataset through cleaning, exploratory analysis, and creation of meaningful new features that capture match dynamics.

In [287]:
ipl_df.shape

(1095, 20)

In [288]:
ipl_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1095 non-null   int64  
 1   season           1095 non-null   int64  
 2   city             1044 non-null   object 
 3   date             1095 non-null   object 
 4   match_type       1095 non-null   object 
 5   player_of_match  1090 non-null   object 
 6   venue            1095 non-null   object 
 7   team1            1095 non-null   object 
 8   team2            1095 non-null   object 
 9   toss_winner      1095 non-null   object 
 10  toss_decision    1095 non-null   object 
 11  winner           1090 non-null   object 
 12  result           1095 non-null   object 
 13  result_margin    1076 non-null   float64
 14  target_runs      1092 non-null   float64
 15  target_overs     1092 non-null   float64
 16  super_over       1095 non-null   object 
 17  method        

In [289]:
ipl_df.describe()

,id,season,result_margin,target_runs,target_overs
count,1.095000e+03,1095.000000,1076.000000,1092.000000,1092.000000
mean,9.048283e+05,2016.126027,17.259294,165.684066,19.759341
std,3.677402e+05,4.946940,21.787444,33.427048,1.581108
min,3.359820e+05,2008.000000,1.000000,43.000000,5.000000
25%,5.483315e+05,2012.000000,6.000000,146.000000,20.000000
50%,9.809610e+05,2016.000000,8.000000,166.000000,20.000000
75%,1.254062e+06,2021.000000,20.000000,187.000000,20.000000
max,1.426312e+06,2024.000000,146.000000,288.000000,20.000000


In [290]:
# Check for missing values in each column
missing_values=ipl_df.isnull().sum().sort_values(ascending=False)
print(missing_values)

method             1074
city                 51
result_margin        19
player_of_match       5
winner                5
target_runs           3
target_overs          3
id                    0
date                  0
season                0
venue                 0
match_type            0
toss_decision         0
toss_winner           0
team2                 0
team1                 0
result                0
super_over            0
umpire1               0
umpire2               0
dtype: int64


In [291]:
#Check for unique values in each column
unique_values=ipl_df.nunique().sort_values(ascending=False)
print(unique_values)

id                 1095
date                823
player_of_match     291
target_runs         170
result_margin        98
umpire1              62
umpire2              62
venue                58
city                 36
toss_winner          19
team1                19
team2                19
winner               19
season               17
target_overs         15
match_type            8
result                4
toss_decision         2
super_over            2
method                1
dtype: int64


In [292]:
ipl_df.value_counts("venue")

,count
venue,
Eden Gardens,77
Wankhede Stadium,73
M Chinnaswamy Stadium,65
Feroz Shah Kotla,60
"Rajiv Gandhi International Stadium, Uppal",49
"MA Chidambaram Stadium, Chepauk",48
Sawai Mansingh Stadium,47
Dubai International Cricket Stadium,46
"Wankhede Stadium, Mumbai",45


In [293]:
#Checking city along with Venue to confirm if some venue names are duplicated here
ipl_df.groupby("venue")["city"].value_counts("venue").sort_values(ascending=False)

,,proportion
venue,city,
Arun Jaitley Stadium,Delhi,1.00
"Arun Jaitley Stadium, Delhi",Delhi,1.00
Barabati Stadium,Cuttack,1.00
"Barsapara Cricket Stadium, Guwahati",Guwahati,1.00
"Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow",Lucknow,1.00
Brabourne Stadium,Mumbai,1.00
"Brabourne Stadium, Mumbai",Mumbai,1.00
Buffalo Park,East London,1.00
De Beers Diamond Oval,Kimberley,1.00


## Observation 1- Multiple Venues are listed duplicated, some are renames and some are duplicated due to city name along with their names, we will standardise this.

In [294]:
ipl_df.value_counts("city")

,count
city,
Mumbai,173
Kolkata,93
Delhi,90
Chennai,85
Hyderabad,77
Bangalore,65
Chandigarh,61
Jaipur,57
Pune,51


## Observation 2 - Some City names are duplicated like Bengaluru and Banglore, we will standardise this as well.

In [295]:
ipl_df.value_counts("team1")

,count
team1,
Royal Challengers Bangalore,135
Chennai Super Kings,128
Mumbai Indians,123
Kolkata Knight Riders,121
Rajasthan Royals,101
Kings XI Punjab,92
Sunrisers Hyderabad,86
Delhi Daredevils,85
Delhi Capitals,41


## Observation 3 - Duplicates are caused here by:
  

*   Some team name changes like Delhi Daredevils to Delhi Capitals, Royal Challengers Bengalore  to Royal Challengers Bengaluru
*   Rising Pune Supergiant and Rising Pune Supergiants , minor spelling change

In [296]:
ipl_df.value_counts("match_type")

,count
match_type,
League,1029
Final,17
Qualifier 1,14
Qualifier 2,14
Eliminator,11
Semi Final,6
Elimination Final,3
3rd Place Play-Off,1


In [297]:
ipl_df.value_counts("method")

,count
method,
D/L,21


## Observation 4- Only one value of method - D/L, we will rather create one column like DL_used - Yes or No

In [298]:
# Null values analysis
null_city_rows = ipl_df[ipl_df['city'].isnull()]
null_method_rows = ipl_df[ipl_df['method'].isnull()]
null_winner_rows = ipl_df[ipl_df['winner'].isnull()]
null_result_margin_rows = ipl_df[ipl_df['result_margin'].isnull()]
null_player_of_match_rows = ipl_df[ipl_df['player_of_match'].isnull()]
null_target_runs_rows = ipl_df[ipl_df['target_runs'].isnull()]
null_target_overs_rows = ipl_df[ipl_df['target_overs'].isnull()]

display("1 ) null_city_rows")
display(null_city_rows)

display("2 ) null_method_rows")
display(null_method_rows)

display("3 ) null_winner_rows")
display(null_winner_rows)

display("4) null_result_margin_rows")
display(null_result_margin_rows)

display("5 ) null_player_of_match_rows")
display(null_player_of_match_rows)

display("6 ) null_target_runs_rows")
display(null_target_runs_rows)

display("7 ) null_target_overs_rows")
display(null_target_overs_rows)

'1 ) null_city_rows'

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
399,729281,2014,NaN,2014-04-17,League,YS Chahal,Sharjah Cricket Stadium,Delhi Daredevils,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Royal Challengers Bangalore,wickets,8.0,146.0,20.0,N,NaN,Aleem Dar,S Ravi
402,729287,2014,NaN,2014-04-19,League,PA Patel,Dubai International Cricket Stadium,Royal Challengers Bangalore,Mumbai Indians,Royal Challengers Bangalore,field,Royal Challengers Bangalore,wickets,7.0,116.0,20.0,N,NaN,Aleem Dar,AK Chaudhary
403,729289,2014,NaN,2014-04-19,League,JP Duminy,Dubai International Cricket Stadium,Kolkata Knight Riders,Delhi Daredevils,Kolkata Knight Riders,bat,Delhi Daredevils,wickets,4.0,167.0,20.0,N,NaN,Aleem Dar,VA Kulkarni
404,729291,2014,NaN,2014-04-20,League,GJ Maxwell,Sharjah Cricket Stadium,Rajasthan Royals,Kings XI Punjab,Kings XI Punjab,field,Kings XI Punjab,wickets,7.0,192.0,20.0,N,NaN,BF Bowden,M Erasmus
406,729295,2014,NaN,2014-04-22,League,GJ Maxwell,Sharjah Cricket Stadium,Kings XI Punjab,Sunrisers Hyderabad,Sunrisers Hyderabad,field,Kings XI Punjab,runs,72.0,194.0,20.0,N,NaN,M Erasmus,S Ravi
407,729297,2014,NaN,2014-04-23,League,RA Jadeja,Dubai International Cricket Stadium,Rajasthan Royals,Chennai Super Kings,Rajasthan Royals,field,Chennai Super Kings,runs,7.0,141.0,20.0,N,NaN,HDPK Dharmasena,RK Illingworth
408,729299,2014,NaN,2014-04-24,League,CA Lynn,Sharjah Cricket Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,2.0,151.0,20.0,N,NaN,Aleem Dar,VA Kulkarni
409,729301,2014,NaN,2014-04-25,League,AJ Finch,Dubai International Cricket Stadium,Sunrisers Hyderabad,Delhi Daredevils,Sunrisers Hyderabad,bat,Sunrisers Hyderabad,runs,4.0,185.0,20.0,N,NaN,M Erasmus,S Ravi
410,729303,2014,NaN,2014-04-25,League,MM Sharma,Dubai International Cricket Stadium,Chennai Super Kings,Mumbai Indians,Mumbai Indians,bat,Chennai Super Kings,wickets,7.0,142.0,20.0,N,NaN,BF Bowden,M Erasmus
413,729309,2014,NaN,2014-04-27,League,M Vijay,Sharjah Cricket Stadium,Delhi Daredevils,Mumbai Indians,Mumbai Indians,bat,Delhi Daredevils,wickets,6.0,126.0,20.0,N,NaN,Aleem Dar,VA Kulkarni


'2 ) null_method_rows'

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2008,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2008,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2008,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2008,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2008,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1090,1426307,2024,Hyderabad,2024-05-19,League,Abhishek Sharma,"Rajiv Gandhi International Stadium, Uppal, Hyd...",Punjab Kings,Sunrisers Hyderabad,Punjab Kings,bat,Sunrisers Hyderabad,wickets,4.0,215.0,20.0,N,NaN,Nitin Menon,VK Sharma
1091,1426309,2024,Ahmedabad,2024-05-21,Qualifier 1,MA Starc,"Narendra Modi Stadium, Ahmedabad",Sunrisers Hyderabad,Kolkata Knight Riders,Sunrisers Hyderabad,bat,Kolkata Knight Riders,wickets,8.0,160.0,20.0,N,NaN,AK Chaudhary,R Pandit
1092,1426310,2024,Ahmedabad,2024-05-22,Eliminator,R Ashwin,"Narendra Modi Stadium, Ahmedabad",Royal Challengers Bengaluru,Rajasthan Royals,Rajasthan Royals,field,Rajasthan Royals,wickets,4.0,173.0,20.0,N,NaN,KN Ananthapadmanabhan,MV Saidharshan Kumar
1093,1426311,2024,Chennai,2024-05-24,Qualifier 2,Shahbaz Ahmed,"MA Chidambaram Stadium, Chepauk, Chennai",Sunrisers Hyderabad,Rajasthan Royals,Rajasthan Royals,field,Sunrisers Hyderabad,runs,36.0,176.0,20.0,N,NaN,Nitin Menon,VK Sharma


'3 ) null_winner_rows'

...

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
241,501265,2011,Delhi,2011-05-21,League,NaN,Feroz Shah Kotla,Delhi Daredevils,Pune Warriors,Delhi Daredevils,bat,NaN,no result,NaN,NaN,NaN,N,NaN,SS Hazare,RJ Tucker
485,829763,2015,Bangalore,2015-04-29,League,NaN,M Chinnaswamy Stadium,Royal Challengers Bangalore,Rajasthan Royals,Rajasthan Royals,field,NaN,no result,NaN,NaN,NaN,N,NaN,JD Cloete,PG Pathak
511,829813,2015,Bangalore,2015-05-17,League,NaN,M Chinnaswamy Stadium,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,field,NaN,no result,NaN,188.0,20.0,N,NaN,HDPK Dharmasena,K Srinivasan
744,1178424,2019,Bengaluru,2019-04-30,League,NaN,M.Chinnaswamy Stadium,Royal Challengers Bangalore,Rajasthan Royals,Rajasthan Royals,field,NaN,no result,NaN,63.0,5.0,N,NaN,NJ Llong,UV Gandhe
994,1359519,2023,Lucknow,2023-05-03,League,NaN,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow Super Giants,Chennai Super Kings,Chennai Super Kings,field,NaN,no result,NaN,NaN,NaN,N,NaN,AK Chaudhary,NA Patwardhan


'6 ) null_target_runs_rows'

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
241,501265,2011,Delhi,2011-05-21,League,NaN,Feroz Shah Kotla,Delhi Daredevils,Pune Warriors,Delhi Daredevils,bat,NaN,no result,NaN,NaN,NaN,N,NaN,SS Hazare,RJ Tucker
485,829763,2015,Bangalore,2015-04-29,League,NaN,M Chinnaswamy Stadium,Royal Challengers Bangalore,Rajasthan Royals,Rajasthan Royals,field,NaN,no result,NaN,NaN,NaN,N,NaN,JD Cloete,PG Pathak
994,1359519,2023,Lucknow,2023-05-03,League,NaN,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow Super Giants,Chennai Super Kings,Chennai Super Kings,field,NaN,no result,NaN,NaN,NaN,N,NaN,AK Chaudhary,NA Patwardhan


'7 ) null_target_overs_rows'

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
241,501265,2011,Delhi,2011-05-21,League,NaN,Feroz Shah Kotla,Delhi Daredevils,Pune Warriors,Delhi Daredevils,bat,NaN,no result,NaN,NaN,NaN,N,NaN,SS Hazare,RJ Tucker
485,829763,2015,Bangalore,2015-04-29,League,NaN,M Chinnaswamy Stadium,Royal Challengers Bangalore,Rajasthan Royals,Rajasthan Royals,field,NaN,no result,NaN,NaN,NaN,N,NaN,JD Cloete,PG Pathak
994,1359519,2023,Lucknow,2023-05-03,League,NaN,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow Super Giants,Chennai Super Kings,Chennai Super Kings,field,NaN,no result,NaN,NaN,NaN,N,NaN,AK Chaudhary,NA Patwardhan


## Observation 5 - Missing Values Explanation:


> a) Method
- 1074 values missing - This is expected as if match was played interrupted ,like rain, only then sometime D/L is applied, else method is null , what we can do maybe  change column to yes/no kind of column, like if_dl.



> b ) city

- 51 values missing - Data Error - Matches were Played in 2014 and 2020 in Sharjah , Abudhabi and Dubai International UAE, we can fill missing values as per city mapping
> c ) result_margin
- 19 values missing - Matches without result - either "Tie"(14) or "no result" (5) won't have any result margin. We can say value is 0 for "Tie" and keep -1 for those 5 "No Results" rows as noise
> d ) player_of_match
- 5 rows - "No result rows"
> e ) winner
- 5 rows - "No result rows" same as above      
>f ) target_runs
- "No result rows" where no overs could be played at all
>g ) target_over
- "No result rows" where no overs could be played at all

## Standardising Columns and Fixing Missing Values

In [299]:
import pandas as pd

# 1. Standardise differently listed Venue Names
standard_venues = {
    # Bengaluru
    "M.Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "M Chinnaswamy Stadium, Bengaluru": "M Chinnaswamy Stadium",

    # Chennai
    "MA Chidambaram Stadium, Chepauk": "MA Chidambaram Stadium",
    "MA Chidambaram Stadium, Chepauk, Chennai": "MA Chidambaram Stadium",

    # Delhi
    "Arun Jaitley Stadium, Delhi": "Arun Jaitley Stadium",
    "Feroz Shah Kotla": "Arun Jaitley Stadium",

    # Mumbai & Navi Mumbai
    "Wankhede Stadium, Mumbai": "Wankhede Stadium",
    "Brabourne Stadium, Mumbai": "Brabourne Stadium",
    "Dr DY Patil Sports Academy, Mumbai": "Dr DY Patil Sports Academy",

    # Chandigarh / Mohali
    "Punjab Cricket Association Stadium, Mohali": "Punjab Cricket Association IS Bindra Stadium",
    "Punjab Cricket Association IS Bindra Stadium, Mohali": "Punjab Cricket Association IS Bindra Stadium",
    "Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh": "Punjab Cricket Association IS Bindra Stadium",

    # Hyderabad
    "Rajiv Gandhi International Stadium, Uppal": "Rajiv Gandhi International Stadium",
    "Rajiv Gandhi International Stadium, Uppal, Hyderabad": "Rajiv Gandhi International Stadium",

    # Pune
    "Maharashtra Cricket Association Stadium, Pune": "Maharashtra Cricket Association Stadium",
    "Subrata Roy Sahara Stadium": "Maharashtra Cricket Association Stadium",

    # Ahmedabad
    "Narendra Modi Stadium, Ahmedabad": "Narendra Modi Stadium",
    "Sardar Patel Stadium, Motera": "Narendra Modi Stadium",

    # Kolkata
    "Eden Gardens, Kolkata": "Eden Gardens",

    # Jaipur
    "Sawai Mansingh Stadium, Jaipur": "Sawai Mansingh Stadium",

    # Dharamsala
    "Himachal Pradesh Cricket Association Stadium, Dharamsala": "Himachal Pradesh Cricket Association Stadium",

    # Visakhapatnam
    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam": "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium",

    # Abu Dhabi
    "Zayed Cricket Stadium, Abu Dhabi": "Sheikh Zayed Stadium",
    "Zayed Cricket Stadium": "Sheikh Zayed Stadium",

    #Comma Separated
    "Barsapara Cricket Stadium, Guwahati": "Barsapara Cricket Stadium",
    "Vidarbha Cricket Association Stadium, Jamtha": "Vidarbha Cricket Association Stadium",
    "Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow": "Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium"
}

# 2. Fix the city inconsistencies alongside the venue clean-up
standard_cities = {
    "Bangalore": "Bengaluru",
    "Navi Mumbai": "Mumbai"
}

# Step A: Clean up the trailing/leading whitespaces
ipl_df['venue'] = ipl_df['venue'].str.strip()
ipl_df['city'] = ipl_df['city'].str.strip()

# Step B: Map the duplicates to their standard names
ipl_df['venue'] = ipl_df['venue'].replace(standard_venues)
ipl_df['city'] = ipl_df['city'].replace(standard_cities)

# Step C: Fix the specific PCA Mohali city classification (from Chandigarh to Mohali)
ipl_df.loc[ipl_df['venue'] == 'Punjab Cricket Association IS Bindra Stadium', 'city'] = 'Mohali'
# Step D: Verifying unique counts now
cleaned_grouped = ipl_df.groupby("venue")["city"].value_counts().sort_values(ascending=False)
print(cleaned_grouped)


venue                                                               city          
Wankhede Stadium                                                    Mumbai            118
M Chinnaswamy Stadium                                               Bengaluru          94
Eden Gardens                                                        Kolkata            93
Arun Jaitley Stadium                                                Delhi              90
MA Chidambaram Stadium                                              Chennai            85
Rajiv Gandhi International Stadium                                  Hyderabad          77
Punjab Cricket Association IS Bindra Stadium                        Mohali             61
Sawai Mansingh Stadium                                              Jaipur             57
Maharashtra Cricket Association Stadium                             Pune               51
Dr DY Patil Sports Academy                                          Mumbai             37
Sheikh Zayed Stad

In [300]:
# 1. Fix Team names inconsistencies  - mapping dictionary for IPL franchises:
standard_teams = {
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}
# 1. Clean team name columns
ipl_df[['team1', 'team2', 'toss_winner', 'winner']] = ipl_df[['team1', 'team2', 'toss_winner', 'winner']].apply(lambda x: x.str.strip().replace(standard_teams))

# 2. Get value counts now
print(ipl_df[['team1', 'team2', 'toss_winner', 'winner']].apply(pd.Series.value_counts).sort_values(by='team1', ascending=False))



                             team1  team2  toss_winner  winner
Royal Challengers Bengaluru    144    111          121     123
Chennai Super Kings            128    110          122     138
Delhi Capitals                 126    126          130     115
Mumbai Indians                 123    138          143     144
Punjab Kings                   123    123          109     112
Kolkata Knight Riders          121    130          122     131
Rajasthan Royals               101    120          120     112
Sunrisers Hyderabad             86     96           88      88
Deccan Chargers                 39     36           43      29
Pune Warriors                   23     23           20      12
Lucknow Super Giants            23     21           19      24
Gujarat Titans                  21     24           22      28
Gujarat Lions                   16     14           15      13
Rising Pune Supergiant          14     16           13      15
Kochi Tuskers Kerala             7      7            8 

In [301]:
# Initialize the venue to city mapping dictionary
venue_to_city_map = {}

# Manually add mappings for the UAE venues that have all missing cities in 2014 or 2020
venue_to_city_map['Sharjah Cricket Stadium'] = 'Sharjah'
venue_to_city_map['Dubai International Cricket Stadium'] = 'Dubai'
venue_to_city_map['Sheikh Zayed Stadium'] = 'Abu Dhabi'

# 2. Fill missing city values using the map
ipl_df['city'] = ipl_df['city'].fillna(ipl_df['venue'].map(venue_to_city_map))

# 3. Verify that there are no more missing values in the 'city' column
missing_cities_count = ipl_df['city'].isnull().sum()
print(f"Missing cities: {missing_cities_count}")

Missing cities: 0


In [302]:
# Convert date column to datetime format
ipl_df['date'] = pd.to_datetime(ipl_df['date'])
ipl_df['date'].dtype

dtype('<M8[ns]')

In [303]:
ipl_df.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2008,Bengaluru,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bengaluru,Kolkata Knight Riders,Royal Challengers Bengaluru,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2008,Mohali,2008-04-19,League,MEK Hussey,Punjab Cricket Association IS Bindra Stadium,Punjab Kings,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2008,Delhi,2008-04-19,League,MF Maharoof,Arun Jaitley Stadium,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,bat,Delhi Capitals,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2008,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,Royal Challengers Bengaluru,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2008,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [304]:
ipl_df.value_counts("result")

,count
result,
wickets,578
runs,498
tie,14
no result,5


In [305]:
# Split result_margin into result_margin_runs and result_margin_wickets as value is very different in case of wickets or run victory
ipl_df['result_margin_runs'] = np.where(
    ipl_df['result'] == "runs",
    ipl_df['result_margin'],
    np.nan
)
ipl_df['result_margin_wickets'] = np.where(
    ipl_df['result'] == "wickets",
    ipl_df['result_margin'],
    np.nan
)

# Display sample of the split columns
print(ipl_df[['result', 'result_margin', 'result_margin_runs', 'result_margin_wickets']].head())
#Drop the result_margin now
ipl_df.drop(columns=['result_margin'], inplace=True)

    result  result_margin  result_margin_runs  result_margin_wickets
0     runs          140.0               140.0                    NaN
1     runs           33.0                33.0                    NaN
2  wickets            9.0                 NaN                    9.0
3  wickets            5.0                 NaN                    5.0
4  wickets            5.0                 NaN                    5.0


### Note - Though this has created NaNs, we need it by design, if I give 0 , it will impact average.

In [306]:
#Handling D/L flag:
ipl_df['is_dl']=np.where(ipl_df['method'].isnull(),0,1)
ipl_df.drop(columns=['method'], inplace=True)
ipl_df['is_dl'].value_counts()

,count
is_dl,
0,1074
1,21


In [307]:
# Compare D/L flag against shortened matches
pd.crosstab(ipl_df['is_dl'], ipl_df['target_overs'] < 20,
            rownames=['is_dl'], colnames=['shortened'])


shortened,False,True
is_dl,,
0,1065,9
1,0,21


In [308]:
ipl_df['is_shortened'] = np.where(ipl_df['target_overs'] < 20, 1, 0)

In [309]:
# match_importance: league / playoff / final
ipl_df['match_importance'] = np.where(
    ipl_df['match_type'] == 'League', 'league',
    np.where(ipl_df['match_type'] == 'Final', 'final', 'playoff')
)
ipl_df['match_importance'].value_counts()

,count
match_importance,
league,1029
playoff,49
final,17


In [310]:
#Assigning Season Phase to Early, Mid or late, idea is to bin them as per their number of days into the season.
# Sort by date and id in case two matches share a date
ipl_df = ipl_df.sort_values(['date', 'id']).reset_index(drop=True)

# Match number within each season
ipl_df['match_number'] = ipl_df.groupby('season').cumcount() + 1

# Split each season into 3 equal-count bins: 0 = early, 1 = mid, 2 = late
phase = ipl_df.groupby('season')['match_number'].transform(
    lambda x: pd.qcut(x, q=3, labels=False)
)
ipl_df['season_phase'] = phase.map({0: 'early', 1: 'mid', 2: 'late'})

# Check: each season should be split roughly into thirds
pd.crosstab(ipl_df['season'], ipl_df['season_phase'])


season_phase,early,late,mid
season,,,
2008,19,20,19
2009,19,19,19
2010,20,20,20
2011,24,25,24
2012,25,25,24
2013,25,26,25
2014,20,20,20
2015,20,20,19
2016,20,20,20


In [311]:
#Toss Advantage - Check if winner and toss winner are same
ipl_df['toss_advantage'] = np.where(
    ipl_df['winner'].isnull(), np.nan,
    np.where(ipl_df['toss_winner'] == ipl_df['winner'], 1, 0)
)

In [312]:
#Home Advantage Check
team_home_city = {
    # Franchises home team mapping some are  Multi-City / Alternate Home Grounds
    "Rajasthan Royals": ["Jaipur", "Guwahati"],
    "Punjab Kings": ["Mohali", "Dharamsala", "Indore", "Cuttack"],
    "Delhi Capitals": ["Delhi", "Visakhapatnam", "Raipur"],
    "Chennai Super Kings": ["Chennai", "Pune", "Ranchi"],
    "Royal Challengers Bengaluru": ["Bengaluru"],
    "Mumbai Indians": ["Mumbai"],
    "Kolkata Knight Riders": ["Kolkata"],
    "Sunrisers Hyderabad": ["Hyderabad"],
    "Lucknow Super Giants": ["Lucknow"],
    "Gujarat Titans": ["Ahmedabad"],
    "Deccan Chargers": ["Hyderabad", "Visakhapatnam"],
    "Gujarat Lions": ["Rajkot", "Kanpur"],
    "Pune Warriors": ["Pune"],
    "Rising Pune Supergiant": ["Pune"],
    "Kochi Tuskers Kerala": ["Kochi"]
}
def is_home(team, city):
    return int(city in team_home_city.get(team, []))

ipl_df['team1_home'] = ipl_df.apply(lambda r: is_home(r['team1'], r['city']), axis=1)
ipl_df['team2_home'] = ipl_df.apply(lambda r: is_home(r['team2'], r['city']), axis=1)


In [313]:
# A match can't be home for both teams
both_home = ipl_df[(ipl_df['team1_home'] == 1) & (ipl_df['team2_home'] == 1)]
both_home[['season', 'city', 'team1', 'team2']]

,season,city,team1,team2
263,2012,Pune,Pune Warriors,Chennai Super Kings
363,2013,Pune,Pune Warriors,Chennai Super Kings


In [314]:
#In 2012 and 2013, Pune was home for Pune Warriors, not for CSK. CSK only used Pune as a home ground in 2018.

#Fix - Treat Pune as a CSK home city only in 2018:
csk_pune_not_home = (ipl_df['city'] == 'Pune') & (ipl_df['season'] != 2018)
ipl_df.loc[csk_pune_not_home & (ipl_df['team1'] == 'Chennai Super Kings'), 'team1_home'] = 0
ipl_df.loc[csk_pune_not_home & (ipl_df['team2'] == 'Chennai Super Kings'), 'team2_home'] = 0

#Check again:
both_home = ipl_df[(ipl_df['team1_home'] == 1) & (ipl_df['team2_home'] == 1)]
both_home[['season', 'city', 'team1', 'team2']]

,season,city,team1,team2


In [315]:
# How many teams were "at home" in each match: 0, 1
home_count = ipl_df['team1_home'] + ipl_df['team2_home']

pd.crosstab(ipl_df['season'], home_count,
            rownames=['season'], colnames=['home_teams'])

home_teams,0,1
season,,
2008,3,55
2009,57,0
2010,11,49
2011,9,64
2012,4,70
2013,6,70
2014,27,33
2015,14,45
2016,6,54


## Observation 6 – Home advantage
Home cities are mapped per franchise, including secondary grounds (e.g. Punjab Kings at Dharamsala/Indore, Rajasthan Royals at Guwahati). A season-specific exception is applied for Chennai Super Kings in Pune, which was a CSK home ground only in 2018. In 2012–13 it belonged to Pune Warriors.
2009 (South Africa), 2020 (UAE) and 2021 are neutral seasons, so no team is marked as home. The 2021 result comes from the data itself, since teams didn't play in their home cities.
2022 was officially held at neutral venues in Mumbai and Pune, but Mumbai Indians' matches in Mumbai are kept as home games, on the assumption that a familiar city and crowd still give some advantage.
Limitation: a small number of league matches may be misclassified where a team used a secondary home ground missing from the mapping. Given how few matches this affects out of 1,095, the impact on the analysis is expected to be minimal.

## Venue Mapping